***PHAASE1***-Input alignment

In [1]:
import pandas as pd

def load_and_clean_data(path):
    df = pd.read_csv(path)

    df.columns = df.columns.str.strip().str.lower()
    df['timestamp'] = pd.to_datetime(df['timestamp'])


    #required fields
    required_cols = ['vessel_id','lat','lon','location','timestamp','signal_type','value']
    df = df[required_cols]
    df = df.dropna()
    df = df.sort_values(by='timestamp').reset_index(drop=True)
    return df

#call
df = load_and_clean_data("marine_data.csv")


# Add NICAI field(imp for taskl 3)
df['signal_id']=df.index.astype(str)
df['status']= 'ALLOW'

df['trace_id']= 'T1'
df['reason']='simulated'

df['confidence_score'] = df['value']/ df['value'].max()

df.head(10)

,vessel_id,lat,lon,location,timestamp,signal_type,value,signal_id,status,trace_id,reason,confidence_score
0,V001,18.96,72.82,Mumbai Coast,2026-01-03,temperature,28.5,0,ALLOW,T1,simulated,0.313187
1,V007,18.96,72.82,Mumbai Coast,2026-01-03,wind_speed,20.0,1,ALLOW,T1,simulated,0.219780
2,V005,15.30,73.90,Goa Coast,2026-01-03,temperature,29.0,2,ALLOW,T1,simulated,0.318681
3,V008,15.00,70.00,Arabian Sea,2026-01-03,oil_spill,30.0,3,ALLOW,T1,simulated,0.329670
4,V003,15.00,70.00,Arabian Sea,2026-01-03,temperature,35.0,4,ALLOW,T1,simulated,0.384615
5,V005,15.30,73.90,Goa Coast,2026-02-03,temperature,30.0,5,ALLOW,T1,simulated,0.329670
6,V008,15.00,70.00,Arabian Sea,2026-02-03,oil_spill,32.0,6,ALLOW,T1,simulated,0.351648
7,V003,15.00,70.00,Arabian Sea,2026-02-03,temperature,34.0,7,ALLOW,T1,simulated,0.373626
8,V001,18.96,72.82,Mumbai Coast,2026-02-03,temperature,30.0,8,ALLOW,T1,simulated,0.329670
9,V007,18.96,72.82,Mumbai Coast,2026-02-03,wind_speed,22.0,9,ALLOW,T1,simulated,0.241758


***Phase-2***- Multi-signal Grouping(core)
creating clusters of signals

In [2]:
df['time_bucket']= df['timestamp'].dt.floor('7D')

used '7D' because the data is monthly so better to use Weekly,can aslo apply 1D Daily & 5D week

In [3]:
grouped = df.groupby(['location','time_bucket'])

now converting groupes in the clusters 

In [4]:
clusters=[]

for name,group in grouped:
    clusters.append(group)


In [5]:
print("Total clusters:",len(clusters))
print(clusters[0])

Total clusters: 30
  vessel_id   lat   lon     location  timestamp  signal_type  value signal_id  \
3      V008  15.0  70.0  Arabian Sea 2026-01-03    oil_spill   30.0         3   
4      V003  15.0  70.0  Arabian Sea 2026-01-03  temperature   35.0         4   

  status trace_id     reason  confidence_score time_bucket  
3  ALLOW       T1  simulated          0.329670  2026-01-01  
4  ALLOW       T1  simulated          0.384615  2026-01-01  


In [6]:
for i, cluster in enumerate(clusters[:3]):
    print(f"\nCluster{i}")
    print(cluster[['signal_type','location','timestamp']])


Cluster0
   signal_type     location  timestamp
3    oil_spill  Arabian Sea 2026-01-03
4  temperature  Arabian Sea 2026-01-03

Cluster1
   signal_type     location  timestamp
6    oil_spill  Arabian Sea 2026-02-03
7  temperature  Arabian Sea 2026-02-03

Cluster2
    signal_type     location  timestamp
10    oil_spill  Arabian Sea 2026-03-03
13  temperature  Arabian Sea 2026-03-03


*in this phase we grouped signals into clusters based on loaction and time window, this alows system to analyze multiple related signals together instead of single*

****Phase3**** *- Multi-Signal Reasoning Engine*

In [7]:
def detect_anomaly(cluster):
    signal_types = set(cluster['signal_type'])

    if 'oil_spill' in signal_types and 'current' in signal_types:
        return "spreed_risk"

    if 'temperature' in signal_types and 'wind' in signal_types:
        return "enivironment_anomaly"

    if 'oil_spill' in signal_types and 'temperature' in signal_types:
        return "marine_anomaly"

    if len(cluster) >=3 and len(signal_types) ==1:
        return "habitual_issue"

    return "normal"

In [8]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)

    results.append({ "anomaly_type": anomaly, "contributing_signals":cluster['signal_type'].tolist()})
    
    

**Check Output**

In [9]:
results[:3]

[{'anomaly_type': 'marine_anomaly',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'contributing_signals': ['oil_spill', 'temperature']}]

*this phase applies rule-based resoning on grouped signals tob detect anomalies based on combinations of signal types*

****Phase4**** *-Temporal context(Trend Detection)*

In [10]:
def detect_trend(cluster):
    cluster = cluster.sort_values(by='timestamp')
    values = cluster['value'].tolist()

    if len(values) < 2:
        return "STABLE"

    if all(x < y for x, y in zip(values, values[1:])):
        return "RISING"
    elif all(x > y for x, y in zip(values, values[1:])):
        return "FALLING"
    else:
        return "STABLE"
        

In [11]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)

    results.append({ "anomaly_type": anomaly, "temporal_context": trend, "contributing_signals": cluster['signal_type'].tolist()})

In [12]:
results[:5]

[{'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'contributing_signals': ['oil_spill', 'temperature']}]

*this phase analyzes tiem_series behaviour of signals within a cluster to determine whether the situation is rising, declining, or stable* 

****Phase5**** *-Spatial Context*

In [13]:
def detect_spatial(cluster):
    locations = cluster['location'].unique().tolist()
    return locations

In [14]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)
    spatial = detect_spatial(cluster)

    results.append({"anomaly_type":anomaly,"temporal_context": trend,"spatial_context": spatial,"contributing_signals": cluster['signal_type'].tolist()})

**Checkoutput**

In [15]:
results[:5]

[{'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'spatial_context': ['Arabian Sea'],
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'spatial_context': ['Arabian Sea'],
  'contributing_signals': ['oil_spill', 'temperature']}]

****Phase6**** *-Confidence layer*

In [16]:
def compute_confidence(cluster):
    avg_conf = cluster['confidence_score'].mean()

    if avg_conf >= 0.75:
        return "HIGH"

    elif avg_conf >=0.5:
        return "MEDIUM"

    else:
        return "LOW"

In [17]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)
    spatial = detect_spatial(cluster)
    confidence = compute_confidence(cluster)

    results.append({"anomaly_type": anomaly, "temporal_context": trend, "spatial_context" : spatial,"confidence": confidence, "contributing_signals": cluster['signal_type'].tolist()})
        

In [18]:
results[:5]

[{'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'MEDIUM',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'MEDIUM',
  'contributing_signals': ['oil_spill', 'temperature']}]

****Phase7**** *- Risk Scoring*

In [19]:
def compute_risk(cluster, confidence):
    unique_signals = cluster['signal_type'].nunique()

    if unique_signals == 1:
        risk_level = "LOW"
    elif unique_signals >= 3:
        risk_level = "HIGH"
    else:
        risk_level = "MEDIUM"

    risk_score = round(unique_signals * 0.3 + 0.4, 2)

    return risk_level, risk_score

    


    

In [20]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)
    spatial = detect_spatial(cluster)
    confidence = compute_confidence(cluster)

    risk_level, risk_score = compute_risk(cluster, confidence)

    results.append({"anomaly_type": anomaly, "temporal_context": trend, "spatial_context" : spatial,"confidence": confidence,"risk_level": risk_level, "risk_score": risk_score, "contributing_signals": cluster['signal_type'].tolist()})
        

In [21]:
results[:5]

[{'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'MEDIUM',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature']},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'FALLI

****Phase8**** *-Explaination Layer*

**create function**

In [22]:
def generate_explanation(cluster, anomaly, trend , spatial, confidence, risk_level):
    signals = ", ".join([s.replace("_"," ") for s in cluster['signal_type'].unique()])
    location = ", ".join(spatial)

    anomaly_text = anomaly.replace("_"," ").title()

    explanation = (
        f"{anomaly_text} detected in {location}"
        f" with {trend.lower()} trend based on {signals} signals,"
        f"confidence is {confidence.lower()} and risk level is {risk_level.lower()}.")

    return explanation

*adding explanation in the loop*

In [23]:
results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)
    spatial = detect_spatial(cluster)
    confidence = compute_confidence(cluster)
    risk_level, risk_score = compute_risk(cluster, confidence)

    explanation = generate_explanation(
        cluster, anomaly, trend, spatial, confidence, risk_level
    )

    results.append({
        "anomaly_type": anomaly,
        "temporal_context": trend,
        "spatial_context": spatial,
        "confidence": confidence,
        "risk_level": risk_level,
        "risk_score": risk_score,
        "contributing_signals": cluster['signal_type'].tolist(),
        "explanation": explanation   # NEW FIELD
    })

In [24]:
results[:2]

[{'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature'],
  'explanation': 'Marine Anomaly detected in Arabian Sea with rising trend based on oil spill, temperature signals,confidence is low and risk level is medium.'},
 {'anomaly_type': 'marine_anomaly',
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'contributing_signals': ['oil_spill', 'temperature'],
  'explanation': 'Marine Anomaly detected in Arabian Sea with rising trend based on oil spill, temperature signals,confidence is low and risk level is medium.'}]

****Phase9**** *-Final Output Contract*

In [25]:
final_results = []

for cluster in clusters:
    anomaly = detect_anomaly(cluster)
    trend = detect_trend(cluster)
    spatial = detect_spatial(cluster)
    risk_level, risk_score = compute_risk(cluster, confidence)

    explanation =generate_explanation( cluster, anomaly, trend, spatial, confidence, risk_level)

    final_results.append({ "situation_summary": f"{anomaly.replace('_',' ').title()}detected",
        "anomaly_type": anomaly,
        "anomaly_count": len(cluster),
        "risk_level": risk_level,
        "risk_score": risk_score,
        "temporal_context": trend,
        "spatial_context": spatial,
        "confidence": confidence,
        "key_indicators": list(cluster['signal_type'].unique()),
        "contributing_signals": cluster['signal_type'].tolist(),
        "explanation": explanation})

In [26]:
final_results[:2]

[{'situation_summary': 'Marine Anomalydetected',
  'anomaly_type': 'marine_anomaly',
  'anomaly_count': 2,
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'key_indicators': ['oil_spill', 'temperature'],
  'contributing_signals': ['oil_spill', 'temperature'],
  'explanation': 'Marine Anomaly detected in Arabian Sea with rising trend based on oil spill, temperature signals,confidence is low and risk level is medium.'},
 {'situation_summary': 'Marine Anomalydetected',
  'anomaly_type': 'marine_anomaly',
  'anomaly_count': 2,
  'risk_level': 'MEDIUM',
  'risk_score': 1.0,
  'temporal_context': 'RISING',
  'spatial_context': ['Arabian Sea'],
  'confidence': 'LOW',
  'key_indicators': ['oil_spill', 'temperature'],
  'contributing_signals': ['oil_spill', 'temperature'],
  'explanation': 'Marine Anomaly detected in Arabian Sea with rising trend based on oil spill, temperature signals,confidence is low

**"This sytem produces a structured intelligence output including anomaly types, risk score, temporal and spatial context, confidence, key indicators, and a human-Readable explanation for decision support".**

***Phase10*** *-determinism check*

**Same input** ****-->**** ********Same output every time********* 

In [27]:
output1 = final_results
output2 = final_results

print(output1 == output2)

True


***Phase11*** *-API Layer.*

 *http://127.0.0.1:8000/docs*

**I build  a deterministic multi-signal intelligence API using fastAPI that processes marine signals and outputs structured anomaly intellignece**